# B2.12 · Black-box agentic pentest — inference, and refusing to report it as fact

**Function B — Application Security with an AI SDLC → The AI SDLC — an Agentic AppSec Pipeline, Before and After Deploy**

Builds on **[B2.11 · White-box agentic pentest — the source, and what it lets you prove](https://spbreed.github.io/cyber-commons/lessons/B2.11.html)**.

| | |
|---|---|
| Tools used | Nuclei |

## What this lesson is

**What it covers.** A black-box engagement: splitting an external probe's claims into what the evidence entails and what it merely suggests, and refusing a severity on the second kind.

**Why a security engineer needs it.** A model with only status codes and headers narrates a confident architecture, and an inference with a CVSS score beside it reads as a finding to everyone downstream. Provenance per claim is the whole discipline, and the open questions it produces tell you which mode to run next.

## 1 · The hook

From outside, an agent will tell you the target runs Django on PostgreSQL with asynchronous refunds, in fluent prose, from a cookie name and a stack trace. Some of that is entailed by the evidence and most of it is a guess, and the report does not say which until you make it.

> **At CyberTravels.** The target is CyberTravels from outside, and the claim the mode cannot reach is the one that matters — whether the refund endpoint accepts a booking it does not own. One account can only ask the question; answering it is a grey-box test.

## 2 · The framework

```
   evidence                 claim                         verdict

   handshake completed      serves TLS 1.3                OBSERVED  -> finding
   200 known / 404 random   /bookings/{id} exists         OBSERVED  -> finding
   404 on a random id       returns ANOTHER tenant's       INFERRED  -> question
   csrftoken cookie         runs Django                    INFERRED  -> question
   1.9s vs 120ms            refunds are synchronous        INFERRED  -> question

   a finding carries a severity. a question carries what would resolve it.
   an inference with a CVSS score is a finding to everyone downstream.
```

Black box is the mode with the least information and the most room to narrate.
An agent handed status codes, headers, error strings and timings will produce a
fluent architecture, and every sentence of it will be plausible. Some of it is
entailed by the evidence. Most of it is not, and the report does not separate
them unless you make it.

The single discipline is provenance per claim. For each thing the agent wants
to assert, keep the actual observation beside it and ask one question: does this
evidence *entail* the claim, or is it merely *consistent* with it?

- A 404 on a random id is consistent with a missing object **and** with correct
  authorisation refusing you. It entails neither.
- A `csrftoken` cookie is consistent with Django and with anything that copied
  its conventions.
- 1.9-second latency on refunds is consistent with synchronous processing and
  with a dozen other causes.

Observed claims are findings. Inferred ones are open questions, and they carry
**no severity** — because an inference with a CVSS number beside it reads as a
finding to everyone downstream, and the person who has to retract it is never
the person who wrote it.

The open questions are not filler. Each one says what mode would resolve it, and
the ones that need a second credential are telling you the engagement should
become grey box (B2.13) rather than a more elaborate black-box guess.

## 3 · The procedure, as a skill

The skill takes twelve claims from an external probe of CyberTravels, splits them on whether the evidence entails or merely suggests each, and reports findings and open questions separately — with a severity on neither guess.

### The skill — [`skills/redteam/blackbox-claim-provenance/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/redteam/blackbox-claim-provenance/SKILL.md)

```yaml
name: blackbox-claim-provenance
description: >-
  Split an external probe's claims into what the evidence entails and what it
  merely suggests, and refuse to attach a severity to the second kind. Use for a
  black-box engagement, or when a report reads more confidently than its
  evidence supports.
allowed-tools: Read, Grep, Glob
```

# An inference with a severity beside it is a finding to everyone downstream

Black box is the mode with the least information and the most room to narrate. A
model handed status codes, headers and timings will produce a fluent
architecture, and every sentence of it will be plausible. Some of it is entailed
by the evidence. Most of it is not, and the report does not distinguish them
unless you make it.

## When to use this

An external engagement with no credential, a bug-bounty triage queue, and any
report where the reader cannot tell which claims were tested.

## Procedure

**1 — List what the mode may look at.** DNS and certificate transparency, the
TLS handshake, HTTP status/headers/body, error strings, response timing, public
artefacts, the scope document. The list is short, and that is the constraint.

**2 — Write each claim with the evidence beside it.** Not a summary of the
evidence — the actual observation that produced the claim.

**3 — Ask one question per claim: does this evidence entail the claim, or is it
merely consistent with it?** A 404 on a random id is consistent with correct
authorisation *and* with a missing object. A `csrftoken` cookie is consistent
with Django and with anything copying its conventions.

**4 — Report observed claims as findings and inferred ones as open questions,
with no severity on the second list.** Then say what each open question would
need. The ones needing a second credential are grey-box tests, not better
black-box ones.

## Example

```
report: 5 findings, 7 open questions
no severity is attached to anything in the second list.
```

The run continues past this. The script is the example: `test_skills.py`
executes it on every build, so this block cannot drift from what the skill
actually prints.

## Output contract

```json
{
  "data_sources": [{"source": "str", "establishes": "str"}],
  "claims": [{"claim": "str", "evidence": "str", "entailed": true}],
  "findings": ["str"],
  "open_questions": [{"claim": "str", "would_need": "str"}]
}
```

## Failure modes

- **Scoring an inference.** A CVSS number converts a guess into a finding for
  every reader after you.
- **Treating untested as disproved.** "No per-account rate limit" from a single
  address is a gap in the test, not a property of the system.
- **Fingerprinting from headers.** They are copied, proxied and templated.
- **Dropping the open questions.** They are the engagement's most useful output:
  they say what mode to run next.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/redteam/blackbox-claim-provenance/scripts/blackbox_claim_provenance.py
SCRIPT = "skills/redteam/blackbox-claim-provenance/scripts/blackbox_claim_provenance.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Five claims are entailed by their evidence and reported as findings; seven are only consistent with it and become open questions with no severity attached. The one that matters — whether the refund endpoint accepts a booking it does not own — is untestable with one account, so it is escalated to a grey-box test rather than guessed at.

## Your turn

Take your last external report and mark every claim observed or inferred. The inferred ones that carry a severity are the ones a client can disprove, and disproving one is how they learn to discount the rest.

---

**Next → [B2.13 · Grey-box agentic pentest — one credential per role, and the matrix it fills](https://spbreed.github.io/cyber-commons/lessons/B2.13.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.12.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.12.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*